In [1]:
import torch

In [2]:
torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=16)

In [3]:
x = torch.tensor([3.0 ,1.0, 4.0], requires_grad=True)
x.register_hook(lambda grad: print("x Grad is:", grad))

In [4]:
gamma = torch.tensor([[0.495404170221900, 0.375434274999402, 0.375434274999402],
        [0.375434274999402, 0.419617426124700, 0.283520366296857],
        [0.375434274999402, 0.283520366296857, 0.419617426124700]], requires_grad = False)

In [5]:
#y = torch.sqrt(x) + x[0] + x[1] + x[2]
y = torch.einsum('...i,...ij->...j', x, gamma)
y2 = (y[..., None] + y[..., None, :])

y2.register_hook(lambda grad: print("y Grad is:", grad))
y2

tensor([[6.7267677713254201, 6.0433856019730445, 6.4516767814565732],
        [6.0433856019730445, 5.3600034326206689, 5.7682946121041976],
        [6.4516767814565732, 5.7682946121041976, 6.1765857915877262]],
       grad_fn=<AddBackward0>)

In [6]:
torch.autograd.grad(y2, x, torch.ones_like(y2))

y Grad is: tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
x Grad is: tensor([7.4776363213242245, 6.4714324045257534, 6.4714324045257534])


(tensor([7.4776363213242245, 6.4714324045257534, 6.4714324045257534]),)

In [7]:
len(x)

3

In [8]:
def finite_diff(func, inp, delta):
    finitediff = 0
    for indx in range(len(inp)):
        new_inp = inp.clone()
        new_inp[indx] += delta
        print(new_inp)
        finitediff += (func(new_inp) - func(inp))/delta
    return finitediff


In [19]:
def funnyfunc(x):
    y = torch.einsum('...i,...ij->...j', x, gamma)
    y2 = (y[..., None] + y[..., None, :])

    y2.register_hook(lambda grad: print("y Grad is:", grad))
    return y2.sum(1)

In [20]:
finite_diff(funnyfunc, x, 1e-10)

tensor([3.0000000001000000, 1.0000000000000000, 4.0000000000000000],
       grad_fn=<CopySlices>)
tensor([3.0000000000000000, 1.0000000001000000, 4.0000000000000000],
       grad_fn=<CopySlices>)
tensor([3.0000000000000000, 1.0000000000000000, 4.0000000001000000],
       grad_fn=<CopySlices>)


tensor([7.1421624170397990, 6.6390981601216481, 6.6392047415320121],
       grad_fn=<AddBackward0>)